<a href="https://colab.research.google.com/github/Manasa23BEIS123/pytorch-deep-learning-projects/blob/main/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("--- 1. Setting up Transfer Learning Model (ResNet-18) ---")

# Load pre-trained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze lower feature extractor layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected classification head for custom classes (e.g., 3 custom categories)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 3)
)

# Simulate standard preprocessed Image Tensors (Batch=150, Channels=3, Height=224, Width=224)
print("--- 2. Preparing Synthetic Image Dataset ---")
X_images = torch.randn(150, 3, 224, 224)
y_labels = torch.tensor(np.random.choice([0, 1, 2], size=150), dtype=torch.long)

train_dataset = TensorDataset(X_images[:120], y_labels[:120])
test_dataset = TensorDataset(X_images[120:], y_labels[120:])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Loss & Optimizer (Only optimizes custom classification layers)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# --- 3. Model Fine-Tuning ---
print("--- 3. Fine-tuning Classification Head ---")
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f}")

# --- 4. Model Evaluation ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\n Transfer Learning Test Accuracy: {accuracy:.2f}%")

--- 1. Setting up Transfer Learning Model (ResNet-18) ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 147MB/s]


--- 2. Preparing Synthetic Image Dataset ---
--- 3. Fine-tuning Classification Head ---
Epoch [1/5] - Loss: 1.1695
Epoch [2/5] - Loss: 1.1085
Epoch [3/5] - Loss: 1.0363
Epoch [4/5] - Loss: 1.0033
Epoch [5/5] - Loss: 0.9822

 Transfer Learning Test Accuracy: 23.33%
